# Demo 3 — Predicted vs measured trip leg

Validate the predictor against a **measured GPS trip leg**: take a leg from `../tests/data/<REG>/`,
predict a duty cycle between the same origin/destination (with via points that keep the predicted route
on the measured one), and compare the speed profile and cumulative fuel consumption.

**Prerequisites**:

- `pip install -e .` in the repository root (conda env `dcp`);
- `HERE_API_KEY` in `.env` (**required**); `SRF_API_KEY` (*optional* — flat gradient without it);
- the sample data in `../tests/data/AY71UCD/` (**not committed** — see `../tests/data/README.md`).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from dcpredictor import (
    DutyCyclePredictor,
    load_default_driving_behavior,
    load_default_vehicle_params,
)

LEG_CSV = Path("../tests/data/AY71UCD/20250227_AY71UCD_Leg1.csv")

## 1. Load the measured leg

The CSV has one row per second of driving (`UnixTime` in ms, `Spd_Kmph_x` speed, `distance_gps`
cumulative distance in m, `FuelRate` in L/hr, `MassKg`, ...). Origin, destination, departure time and
vehicle mass are all taken from the measurement itself.

In [ ]:
measured_df = pd.read_csv(LEG_CSV)
measured_df["timestamp"] = pd.to_datetime(measured_df["UnixTime"], unit="ms")

origin = (measured_df.iloc[0]["Latitude"], measured_df.iloc[0]["Longitude"])
destination = (measured_df.iloc[-1]["Latitude"], measured_df.iloc[-1]["Longitude"])
departure_time = measured_df.iloc[0]["timestamp"].to_pydatetime()

mass_kg = float(measured_df["MassKg"].dropna().mean())
if np.isnan(mass_kg):
    mass_kg = 5000.0  # fallback when the leg carries no mass signal

# Measured cumulative fuel: integrate FuelRate (L/hr) over the time step.
dt_hr = measured_df["timestamp"].diff().dt.total_seconds().fillna(0) / 3600.0
measured_df["fuel_cumulative_L"] = (measured_df["FuelRate"] * dt_hr).cumsum()

print(f"Leg            : {LEG_CSV.name}")
print(f"Origin         : ({origin[0]:.4f}, {origin[1]:.4f})")
print(f"Destination    : ({destination[0]:.4f}, {destination[1]:.4f})")
print(f"Departure      : {departure_time}")
print(f"Distance       : {measured_df['distance_gps'].iloc[-1] / 1000:.1f} km")
print(f"Duration       : {measured_df['timestamp'].iloc[-1] - measured_df['timestamp'].iloc[0]}")
print(f"Avg mass       : {mass_kg:.0f} kg")
print(f"Measured fuel  : {measured_df['fuel_cumulative_L'].iloc[-1]:.2f} L")

## 2. Via points

Origin and destination alone let the router pick any route. A handful of evenly-spaced intermediate
waypoints from the measured trajectory keeps the predicted route on the road the vehicle actually took.
(If the routed and measured trajectories still diverge, increase `num_points`.)

In [ ]:
def make_via_points(df: pd.DataFrame, num_points: int = 10) -> list:
    """Evenly-spaced intermediate waypoints (origin/destination excluded)."""
    idx = np.linspace(0, len(df) - 1, num_points + 2).round().astype(int)[1:-1]
    return [(df.iloc[i]["Latitude"], df.iloc[i]["Longitude"]) for i in idx]


via_points = make_via_points(measured_df, num_points=10)
via_points

## 3. Predict the duty cycle for the same trip

In [ ]:
predictor = DutyCyclePredictor()  # reads API keys from .env

result = predictor.predict(
    origin=origin,
    destination=destination,
    mass_kg=mass_kg,
    vehicle_params=load_default_vehicle_params("AY71UCD"),
    driving_behavior=load_default_driving_behavior(),
    departure_time=departure_time,
    via_points=via_points,
)

assert result is not None, "Route too short or invalid - no duty cycle generated."

# The three profiles are row-aligned; carry the cumulative fuel onto the speed profile.
pred_df = result.speed_profile.copy()
pred_df["fuel_cumulative_L"] = result.energy_profile["fuel_cumulative_L"].to_numpy()
print(f"Predicted: {len(pred_df)} steps, {pred_df['distance'].iloc[-1] / 1000:.1f} km, "
      f"{pred_df['fuel_cumulative_L'].iloc[-1]:.2f} L")

## 4. Compare speed profiles

In [ ]:
fig, (ax_time, ax_dist) = plt.subplots(2, 1, figsize=(12, 6))

ax_time.plot(measured_df["timestamp"], measured_df["Spd_Kmph_x"], color="red", lw=1, label="Measured")
ax_time.plot(pred_df["timestamp"], pred_df["speed"] * 3.6, color="blue", lw=1, label="Predicted")
ax_time.set_xlabel("Time")
ax_time.set_ylabel("Speed (km/h)")
ax_time.legend()
ax_time.grid(True)
ax_time.set_title("Speed vs time")

ax_dist.plot(measured_df["distance_gps"] / 1000, measured_df["Spd_Kmph_x"], color="red", lw=1, label="Measured")
ax_dist.plot(pred_df["distance"] / 1000, pred_df["speed"] * 3.6, color="blue", lw=1, label="Predicted")
ax_dist.set_xlabel("Distance (km)")
ax_dist.set_ylabel("Speed (km/h)")
ax_dist.legend()
ax_dist.grid(True)
ax_dist.set_title("Speed vs distance")

plt.tight_layout()
plt.show()

## 5. Compare cumulative fuel consumption

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(measured_df["distance_gps"] / 1000, measured_df["fuel_cumulative_L"], color="red", lw=2, label="Measured")
ax.plot(pred_df["distance"] / 1000, pred_df["fuel_cumulative_L"], color="blue", lw=2, label="Predicted")
ax.set_xlabel("Distance (km)")
ax.set_ylabel("Cumulative fuel (L)")
ax.legend()
ax.grid(True)
ax.set_title("Cumulative fuel vs distance")
plt.tight_layout()
plt.show()

fuel_measured_L = measured_df["fuel_cumulative_L"].iloc[-1]
fuel_predicted_L = pred_df["fuel_cumulative_L"].iloc[-1]
error_pct = (fuel_predicted_L - fuel_measured_L) / fuel_measured_L * 100
print(f"Measured : {fuel_measured_L:.2f} L")
print(f"Predicted: {fuel_predicted_L:.2f} L")
print(f"Error    : {error_pct:+.1f} %")

## 6. Fuel error per distance interval

In [ ]:
INTERVAL_M = 20_000  # 20 km bins

max_distance_m = measured_df["distance_gps"].iloc[-1]
edges = np.arange(0, max_distance_m, INTERVAL_M)
edges = np.append(edges, max_distance_m)


def fuel_at(df: pd.DataFrame, dist_col: str, upto_m: float) -> float:
    """Cumulative fuel at the last sample within `upto_m` metres."""
    subset = df[df[dist_col] <= upto_m]
    return float(subset["fuel_cumulative_L"].iloc[-1]) if not subset.empty else np.nan


stats = pd.DataFrame(
    {
        "Distance (km)": (edges[1:] / 1000).round(1),
        "Measured (L)": [fuel_at(measured_df, "distance_gps", e) for e in edges[1:]],
        "Predicted (L)": [fuel_at(pred_df, "distance", e) for e in edges[1:]],
    }
)
stats["Error (%)"] = ((stats["Predicted (L)"] - stats["Measured (L)"]) / stats["Measured (L)"] * 100).round(1)
stats.round(2)

## Notes

- Any other leg under `../tests/data/<REG>/` works — change `LEG_CSV` at the top.
- The default driving-behaviour preset is not calibrated per driver; systematic speed-profile deviations
  can be reduced by tuning `DrivingBehavior` (e.g. `v_cruise`, `a_acc`, `a_dec`).
- Without an `SRF_API_KEY` the gradient falls back to flat, which typically shifts the fuel estimate on
  hilly routes.